<a href="https://colab.research.google.com/github/AliAI11/DolphinMind/blob/main/notebooks/04_final_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers bitsandbytes accelerate rouge-score sentence-transformers faiss-cpu

print("all dependencies installed")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 87.0 MB/s eta 0:00:00
all dependencies installed


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from rouge_score import rouge_scorer
import time
import psutil
import json
import numpy as np

print("imports successful")

imports successful


In [3]:
import urllib.request

url = "https://www.gutenberg.org/files/14833/14833-0.txt"

with urllib.request.urlopen(url) as response:
    long_context = response.read().decode('utf-8')

start_marker = "*** START OF"
end_marker = "*** END OF"

if start_marker in long_context:
    long_context = long_context.split(start_marker)[1]
if end_marker in long_context:
    long_context = long_context.split(end_marker)[0]

test_queries = [
    "Who is Sir Francis Varney?",
    "What happens to Flora Bannerworth?",
    "Who is Admiral Bell and what role does he play?"
]

reference_answers = [
    "Sir Francis Varney is the mysterious vampyre who terrorizes the Bannerworth family.",
    "Flora Bannerworth is attacked by the vampyre and her family tries to protect her from further harm.",
    "Admiral Bell is a friend who helps the Bannerworth family fight against the vampyre threat."
]

print(f"loaded document: {len(long_context.split()):,} words")
print(f"test queries: {len(test_queries)}")

loaded document: 329,160 words
test queries: 3


In [4]:
print("\n" + "="*60)
print("loading smollm3-3b with yarn for extended context")
print("="*60 + "\n")

from transformers import AutoConfig

model_name_smol = "HuggingFaceTB/SmolLM3-3B"

# load and modify config BEFORE loading model
config = AutoConfig.from_pretrained(model_name_smol, trust_remote_code=True)

# configure yarn for 128k context (must be done before model load)
config.rope_scaling = {
    "factor": 2.0,  # 2x65536 = 131,072 tokens
    "original_max_position_embeddings": 65536,
    "type": "yarn"
}
config.max_position_embeddings = 131072

print(f"configured yarn scaling: {config.rope_scaling}")
print(f"max context: {config.max_position_embeddings:,} tokens")

# 4-bit quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"\nloading {model_name_smol} with modified config...")

# load model with yarn-enabled config
model_smol = AutoModelForCausalLM.from_pretrained(
    model_name_smol,
    config=config,  # critical: pass modified config
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer_smol = AutoTokenizer.from_pretrained(model_name_smol, trust_remote_code=True)

print(f"smollm3 loaded successfully")
print(f"configured max context: {model_smol.config.max_position_embeddings:,} tokens")
print(f"ram usage: {psutil.virtual_memory().used / 1e9:.2f} gb\n")


loading smollm3-3b with yarn for extended context



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

configured yarn scaling: {'factor': 2.0, 'original_max_position_embeddings': 65536, 'type': 'yarn'}
max context: 131,072 tokens

loading HuggingFaceTB/SmolLM3-3B with modified config...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

smollm3 loaded successfully
configured max context: 131,072 tokens
ram usage: 5.03 gb



In [5]:
def ask_smollm(context, query, max_context_tokens=4000):
    """generate response using smollm3 with proper context truncation"""

    # truncate context to specified length (prevents tokenizer warning)
    context_tokens = tokenizer_smol.encode(
        context,
        add_special_tokens=False,
        truncation=True,
        max_length=max_context_tokens
    )
    truncated_context = tokenizer_smol.decode(context_tokens, skip_special_tokens=True)

    # build messages using chat format
    messages = [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": f"Context: {truncated_context}\n\nQuestion: {query}\n\nAnswer:"}
    ]

    # apply chat template
    prompt = tokenizer_smol.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # tokenize
    inputs = tokenizer_smol(
        prompt,
        return_tensors="pt",
        truncation=False
    ).to(model_smol.device)

    actual_tokens = inputs['input_ids'].shape[1]

    # generate with recommended parameters
    outputs = model_smol.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.95,
        do_sample=True
    )

    # extract only the generated portion
    generated_ids = outputs[0][len(inputs.input_ids[0]):]
    answer = tokenizer_smol.decode(generated_ids, skip_special_tokens=True)

    return answer.strip(), actual_tokens

print("inference function defined")

inference function defined


In [6]:
print("\n" + "="*60)
print("testing smollm3 with varying context lengths")
print("comparing native long context vs semantic retrieval")
print("="*60 + "\n")

# testing 4k to 128k with proper yarn configuration
context_sizes = [4000, 8000, 16000, 32000, 64000, 128000]
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

smollm_results = []

for max_tokens in context_sizes:
    print(f"\n{'='*50}")
    print(f"testing with {max_tokens:,} token context")
    print(f"{'='*50}")

    predictions = []
    times = []
    actual_tokens_used = []

    for i, (query, ref) in enumerate(zip(test_queries, reference_answers)):
        print(f"query {i+1}: {query}")

        start = time.time()
        answer, tokens = ask_smollm(long_context, query, max_context_tokens=max_tokens)
        elapsed = time.time() - start

        times.append(elapsed)
        actual_tokens_used.append(tokens)
        predictions.append(answer)

        print(f"  time: {elapsed:.2f}s")
        print(f"  tokens: {tokens:,}")
        print(f"  answer: {answer}")

    # calculate rouge-l scores
    rouge_scores = [
        scorer.score(ref, pred)['rougeL'].fmeasure
        for pred, ref in zip(predictions, reference_answers)
    ]

    avg_rouge = np.mean(rouge_scores)
    avg_time = np.mean(times)
    avg_tokens = np.mean(actual_tokens_used)

    result = {
        'method': f'SmolLM3 ({max_tokens//1000}k)',
        'context_size': max_tokens,
        'rouge_l': avg_rouge,
        'avg_time': avg_time,
        'avg_tokens': avg_tokens,
        'ram_gb': psutil.virtual_memory().used / 1e9
    }

    smollm_results.append(result)

    print(f"\nresults for {max_tokens//1000}k context:")
    print(f"  rouge-l: {avg_rouge:.3f}")
    print(f"  avg time: {avg_time:.2f}s")
    print(f"  avg tokens: {avg_tokens:,.0f}")
    print(f"  ram: {result['ram_gb']:.2f} gb")

print("\nsmollm3 testing complete\n")


testing smollm3 with varying context lengths
comparing native long context vs semantic retrieval


testing with 4,000 token context
query 1: Who is Sir Francis Varney?
  time: 11.59s
  tokens: 4,083
  answer: Based on the context provided, Sir Francis Varney is the central figure in the story of "Varney the Vampyre" and is described as a mysterious and potentially supernatural individual. He is the host of the Bannersworth Hall, which is the main location of the events in the story. Sir Francis Varney is also the one who is suspected to be the vampire, feeding on the blood of others, as evidenced by his willingness to offer assistance to the vampire, and his involvement in the various encounters and challenges faced by the characters in the story.

However, the exact nature and identity of Sir Francis Varney remain ambiguous, as the story is written in a way that leaves room for interpretation. Some readers might view him as a man of
query 2: What happens to Flora Bannerworth?
  time:

In [11]:
def calculate_token_usage(method_name, context, queries,
                          chunk_size=500, overlap=100, top_k=5):

    """calculate total tokens processed by each method"""
    tokenizer_qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

    # encode with truncation to avoid warning
    full_context_tokens = len(tokenizer_qwen.encode(
        context,
        truncation=True,
        max_length=500000
    ))
    words = context.split()

    result = {
        'method': method_name,
        'full_doc_tokens': full_context_tokens,
        'tokens_per_query': [],
        'total_tokens': 0,
        'embedding_tokens': 0
    }

    if method_name == "Full Context (Hypothetical)":
        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = full_context_tokens + query_tokens + 100
            result['tokens_per_query'].append(tokens)
        result['total_tokens'] = sum(result['tokens_per_query'])

    elif method_name == "Truncated Context":
        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = 4000 + query_tokens + 100
            result['tokens_per_query'].append(tokens)
        result['total_tokens'] = sum(result['tokens_per_query'])

    elif method_name == "Naive Chunking":
        num_chunks = len(words) // chunk_size
        avg_chunk_tokens = full_context_tokens // num_chunks if num_chunks > 0 else full_context_tokens

        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = (avg_chunk_tokens * 3) + query_tokens + 100
            result['tokens_per_query'].append(tokens)

        result['total_tokens'] = sum(result['tokens_per_query'])
        result['embedding_tokens'] = full_context_tokens

    elif method_name == "DolphinMind RAG":
        estimated_chunks = int((len(words) - chunk_size) / (chunk_size - overlap)) + 1
        avg_chunk_tokens = full_context_tokens // (len(words) // chunk_size)

        for query in queries:
            query_tokens = len(tokenizer_qwen.encode(query))
            tokens = (avg_chunk_tokens * top_k) + query_tokens + 100
            result['tokens_per_query'].append(tokens)

        result['total_tokens'] = sum(result['tokens_per_query'])
        result['embedding_tokens'] = full_context_tokens

    return result

print("token analysis function defined")

token analysis function defined


In [19]:
from transformers import AutoTokenizer

# Initialize tokenizer once
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

print("\n" + "="*60)
print("token efficiency analysis")
print("="*60 + "\n")

def calculate_token_usage(method, context, queries):
    """calculate approximate token usage for each method"""

    if method == "Full Context (Hypothetical)":
        # every query would use the full context
        context_tokens = len(tokenizer.encode(context, truncation=False))
        total = context_tokens * len(queries)
        return {
            'method': method,
            'total_tokens': total,
            'embedding_tokens': 0
        }

    elif method == "Truncated Context":
        # fixed 4000 tokens per query
        total = 4000 * len(queries)
        return {
            'method': method,
            'total_tokens': total,
            'embedding_tokens': 0
        }

    elif method == "Naive Chunking":
        # ~2800 tokens per query (from your results)
        # plus one-time embedding cost
        context_tokens = len(tokenizer.encode(context, truncation=False))
        total = (2800 * len(queries))
        return {
            'method': method,
            'total_tokens': total,
            'embedding_tokens': context_tokens
        }

    elif method == "DolphinMind RAG":
        # ~3500 tokens per query (from your results)
        # plus one-time embedding cost
        context_tokens = len(tokenizer.encode(context, truncation=False))
        total = (3500 * len(queries))
        return {
            'method': method,
            'total_tokens': total,
            'embedding_tokens': context_tokens
        }

methods = [
    "Full Context (Hypothetical)",
    "Truncated Context",
    "Naive Chunking",
    "DolphinMind RAG"
]

token_results = []

print("calculating token usage for each method...\n")

for method in methods:
    result = calculate_token_usage(method, long_context, test_queries)
    token_results.append(result)

    print(f"{method}:")
    print(f"  total tokens processed: {result['total_tokens']:,}")
    print(f"  tokens per query: {result['total_tokens'] // len(test_queries):,}")
    if result['embedding_tokens'] > 0:
        print(f"  embedding overhead (one-time): {result['embedding_tokens']:,}")
    print()

print("token efficiency comparison:")
full_tokens = token_results[0]['total_tokens']
for result in token_results[1:]:
    reduction = ((full_tokens - result['total_tokens']) / full_tokens) * 100
    print(f"  {result['method']}: {reduction:.1f}% fewer tokens vs full context")

print("\ntoken analysis complete\n")



token efficiency analysis

calculating token usage for each method...



Token indices sequence length is longer than the specified maximum sequence length for this model (453222 > 131072). Running this sequence through the model will result in indexing errors


Full Context (Hypothetical):
  total tokens processed: 1,359,666
  tokens per query: 453,222

Truncated Context:
  total tokens processed: 12,000
  tokens per query: 4,000

Naive Chunking:
  total tokens processed: 8,400
  tokens per query: 2,800
  embedding overhead (one-time): 453,222

DolphinMind RAG:
  total tokens processed: 10,500
  tokens per query: 3,500
  embedding overhead (one-time): 453,222

token efficiency comparison:
  Truncated Context: 99.1% fewer tokens vs full context
  Naive Chunking: 99.4% fewer tokens vs full context
  DolphinMind RAG: 99.2% fewer tokens vs full context

token analysis complete



In [20]:
print("\n" + "="*60)
print("final comparison: all methods")
print("="*60 + "\n")

# Updated results from your actual experiments
all_results = [
    # Qwen2.5-3B results (UPDATED)
    {'method': 'DolphinMind (Qwen)', 'rouge_l': 0.185, 'time': 22.54, 'tokens': 3500, 'approach': 'Semantic RAG'},
    {'method': 'RLM-Tools (Qwen)', 'rouge_l': 0.154, 'time': 7.24, 'tokens': 2100, 'approach': 'Tool Calling'},
    {'method': 'Truncated (Qwen)', 'rouge_l': 0.146, 'time': 7.79, 'tokens': 4000, 'approach': 'Baseline'},
    {'method': 'Naive Chunking (Qwen)', 'rouge_l': 0.135, 'time': 6.31, 'tokens': 2800, 'approach': 'TF-IDF Retrieval'},
    {'method': 'Hierarchical (Qwen)', 'rouge_l': 0.117, 'time': 117.53, 'tokens': 52000, 'approach': 'Hierarchical Sum.'},
    {'method': 'Map-Reduce (Qwen)', 'rouge_l': 0.116, 'time': 101.26, 'tokens': 45000, 'approach': 'Summarization'},

    # SmolLM3 native long context results
    {'method': 'SmolLM3 (64k)', 'rouge_l': 0.154, 'time': 12.95, 'tokens': 64085, 'approach': 'Native Long Context'},
    {'method': 'SmolLM3 (16k)', 'rouge_l': 0.141, 'time': 9.76, 'tokens': 16085, 'approach': 'Native Long Context'},
    {'method': 'SmolLM3 (4k)', 'rouge_l': 0.133, 'time': 10.53, 'tokens': 4085, 'approach': 'Native Long Context'},
    {'method': 'SmolLM3 (128k)', 'rouge_l': 0.132, 'time': 26.47, 'tokens': 128085, 'approach': 'Native Long Context'},
    {'method': 'SmolLM3 (32k)', 'rouge_l': 0.122, 'time': 10.29, 'tokens': 32085, 'approach': 'Native Long Context'},
    {'method': 'SmolLM3 (8k)', 'rouge_l': 0.112, 'time': 9.88, 'tokens': 8085, 'approach': 'Native Long Context'},
]

# sort by rouge-l
all_results_sorted = sorted(all_results, key=lambda x: x['rouge_l'], reverse=True)

print(f"{'Method':<30} {'ROUGE-L':>10} {'Time(s)':>10} {'Tokens':>12} {'Approach':>20}")
print("="*85)

for r in all_results_sorted:
    print(f"{r['method']:<30} {r['rouge_l']:>10.3f} {r['time']:>10.2f} "
          f"{r['tokens']:>12,.0f} {r['approach']:>20}")

print("="*85)


final comparison: all methods

Method                            ROUGE-L    Time(s)       Tokens             Approach
DolphinMind (Qwen)                  0.185      22.54        3,500         Semantic RAG
RLM-Tools (Qwen)                    0.154       7.24        2,100         Tool Calling
SmolLM3 (64k)                       0.154      12.95       64,085  Native Long Context
Truncated (Qwen)                    0.146       7.79        4,000             Baseline
SmolLM3 (16k)                       0.141       9.76       16,085  Native Long Context
Naive Chunking (Qwen)               0.135       6.31        2,800     TF-IDF Retrieval
SmolLM3 (4k)                        0.133      10.53        4,085  Native Long Context
SmolLM3 (128k)                      0.132      26.47      128,085  Native Long Context
SmolLM3 (32k)                       0.122      10.29       32,085  Native Long Context
Hierarchical (Qwen)                 0.117     117.53       52,000    Hierarchical Sum.
Map-Reduce 

In [22]:
print("\n" + "="*60)
print("key findings")
print("="*60 + "\n")

best = all_results_sorted[0]
dolphin = [r for r in all_results if 'DolphinMind' in r['method']][0]
smollm_results = [r for r in all_results if 'SmolLM3' in r['method']]
best_smol = max(smollm_results, key=lambda x: x['rouge_l'])

print("best overall:")
print(f"  {best['method']}: {best['rouge_l']:.3f} ROUGE-L")
print()

print("dolphinmind rag:")
print(f"  rouge-l: {dolphin['rouge_l']:.3f}")
print(f"  tokens: {dolphin['tokens']:,.0f}")
print(f"  time: {dolphin['time']:.2f}s")
print(f"  ram overhead: ~0.46 GB (embedding + FAISS)")
print()

print(f"best native long context ({best_smol['method']}):")
print(f"  rouge-l: {best_smol['rouge_l']:.3f}")
print(f"  tokens: {best_smol['tokens']:,.0f}")
print(f"  time: {best_smol['time']:.2f}s")
print()

accuracy_diff = ((dolphin['rouge_l'] - best_smol['rouge_l']) / best_smol['rouge_l']) * 100
token_reduction = ((best_smol['tokens'] - dolphin['tokens']) / best_smol['tokens'] * 100)

print(f"dolphinmind vs best native long context:")
print(f"  accuracy gain: {accuracy_diff:+.1f}%")
print(f"  token reduction: {token_reduction:.1f}%")
print()

# calculate full document size
full_doc_tokens = len(tokenizer.encode(long_context, truncation=False))

print("efficiency:")
print(f"  full doc: {full_doc_tokens:,} tokens")
print(f"  dolphinmind: {dolphin['tokens']:,.0f} tokens/query")
print(f"  reduction: {((full_doc_tokens - dolphin['tokens']) / full_doc_tokens * 100):.1f}%")
print()

print("key insights:")
print(f"  1. DolphinMind achieves BEST accuracy ({dolphin['rouge_l']:.3f}) ")
print(f"     - Beats SmolLM3 @ 64k by {accuracy_diff:+.1f}%")
print(f"     - Beats all other Qwen methods")
print(f"  2. {token_reduction:.1f}% fewer tokens than 64k native context")
print(f"  3. RAM overhead: only ~0.46 GB (embedding + FAISS)")
print(f"     - Peak operation: ~5.84 GB total")
print(f"     - Fits comfortably in Google Colab free tier")
print(f"  4. Naive chunking is fastest ({all_results[3]['time']:.2f}s)")
print(f"     - But 27% less accurate than DolphinMind")
print(f"  5. Native long context doesn't guarantee better performance:")
print(f"     - SmolLM3 @ 64k: 0.154 ROUGE-L")
print(f"     - SmolLM3 @ 128k: 0.132 ROUGE-L (14% WORSE!)")
print(f"  6. Semantic retrieval > Context window size")

print("\n" + "="*60)
print("ram breakdown (dolphinmind)")
print("="*60)
print(f"  Base system (Qwen 4-bit):     5.28 GB")
print(f"  + Embedding model:            0.06 GB")
print(f"  + FAISS index (peak):         0.50 GB")
print(f"  = Total peak RAM:             5.84 GB")
print(f"  DolphinMind overhead only:    0.46 GB ")
print("="*60)

print("\n" + "="*60)
print("analysis complete")
print("="*60)


key findings

best overall:
  DolphinMind (Qwen): 0.185 ROUGE-L

dolphinmind rag:
  rouge-l: 0.185
  tokens: 3,500
  time: 22.54s
  ram overhead: ~0.46 GB (embedding + FAISS)

best native long context (SmolLM3 (64k)):
  rouge-l: 0.154
  tokens: 64,085
  time: 12.95s

dolphinmind vs best native long context:
  accuracy gain: +20.1%
  token reduction: 94.5%

efficiency:
  full doc: 453,222 tokens
  dolphinmind: 3,500 tokens/query
  reduction: 99.2%

key insights:
  1. DolphinMind achieves BEST accuracy (0.185) 
     - Beats SmolLM3 @ 64k by +20.1%
     - Beats all other Qwen methods
  2. 94.5% fewer tokens than 64k native context
  3. RAM overhead: only ~0.46 GB (embedding + FAISS)
     - Peak operation: ~5.84 GB total
     - Fits comfortably in Google Colab free tier
  4. Naive chunking is fastest (6.31s)
     - But 27% less accurate than DolphinMind
  5. Native long context doesn't guarantee better performance:
     - SmolLM3 @ 64k: 0.154 ROUGE-L
     - SmolLM3 @ 128k: 0.132 ROUGE-L (

In [18]:
results_package = {
    'smollm3_results': smollm_results,
    'token_analysis': token_results,
    'final_comparison': all_results_sorted,
    'metadata': {
        'document_words': len(long_context.split()),
        'document_tokens': full_doc_tokens,
        'num_queries': len(test_queries),
        'best_method': best['method'],
        'best_rouge_l': best['rouge_l']
    }
}

with open('final_evaluation_results.json', 'w') as f:
    json.dump(results_package, f, indent=2)

print("\n" + "="*60)
print("evaluation complete - results saved")
print("="*60)


evaluation complete - results saved
